# 06 · Eval（召回 → 排序 → RAGAS）

这节的目标是把评估讲清楚：

1) **评估召回**（Retrieval）：Recall@k / Hit@k（先不谈生成）
2) **评估排序**（Ranking）：MRR / nDCG（看 top-k 里谁排前面）
3) **评估端到端**（RAGAS）：把检索 + 生成放在一起评估

> 说明：为了课堂可跑、可复现，这里用一个很小的 query 集合；相关性标注用 LLM 做“弱监督评估”，你也可以替换成手工标注。


In [1]:
pip install ragas

Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import json
import math
import os
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()

    for candidate in (cwd, cwd.parent):
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate

    raise FileNotFoundError("未找到项目根目录，请从 RAG_project 根目录或 notebooks 目录运行本 notebook。")



def load_project_env(project_root: Path) -> Path | None:
    for env_path in (project_root / ".env", project_root.parent / ".env"):
        if env_path.exists():
            load_dotenv(env_path, override=True)
            return env_path
    return None


PROJECT_ROOT = resolve_project_root()
ENV_FILE = load_project_env(PROJECT_ROOT)
CHROMA_DIR = PROJECT_ROOT / "data/chroma"
COLLECTION = "autel_annual_report_2024"

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_base_url = os.getenv("OPENAI_BASE_URL", "https://openrouter.ai/api/v1")
embed_model = os.getenv("EMBED_MODEL", "text-embedding-3-small")
chat_model = os.getenv("CHAT_MODEL") or os.getenv("LLM_MODEL", "gpt-4o-mini")

assert CHROMA_DIR.exists(), f"找不到 Chroma 目录：{CHROMA_DIR.resolve()}（先跑 01_data_02_chunk_ingest.ipynb）"
assert openai_api_key, f"未加载 OPENAI_API_KEY（检查 {ENV_FILE or PROJECT_ROOT.parent / '.env'}）"

client_kwargs = {
    "api_key": openai_api_key,
    "base_url": openai_base_url,
}

# 当前本地 Chroma 已按 1536 维 embedding 建库；若更换 embedding 模型，请先重建 data/chroma。
emb = OpenAIEmbeddings(model=embed_model, **client_kwargs)
vs = Chroma(collection_name=COLLECTION, embedding_function=emb, persist_directory=str(CHROMA_DIR))
llm = ChatOpenAI(model=chat_model, temperature=0, **client_kwargs)

print("ready:", COLLECTION, "embed:", embed_model, "chat:", chat_model, "env:", ENV_FILE)


/opt/anaconda3/envs/voc/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/var/folders/hb/4k5shxzs4c7by7lm5s0mmgrw0000gn/T/ipykernel_7870/2676058162.py:56: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vs = Chroma(collection_name=COLLECTION, embedding_function=emb, persist_directory=str(CHROMA_DIR))


ready: autel_annual_report_2024 embed: text-embedding-3-small chat: openai/gpt-5.4 env: /Users/mengbai/Documents/AI-training/.env


In [ ]:

EVAL_QUERIES = [
    "道通2024年年报里，主营业务/产品线的收入结构如何？",
    "年报里研发投入的主要方向是什么？",
    "年报里提到的海外/国际化业务有哪些信息？",
    "年报里有哪些风险因素或风险提示？",
]

TOPK = 8


def parse_json_object(raw: str, fallback: dict):
    text = (raw or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.DOTALL).strip()

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        text = match.group(0)

    try:
        data = json.loads(text)
        if isinstance(data, dict):
            return data
    except Exception:
        pass

    return {**fallback, "_raw": (raw or "")[:240]}


def retrieve(query: str, k: int = TOPK):
    return vs.similarity_search(query, k=k)


In [4]:
# 1) 相关性判定器（LLM judge）：给 query + chunk 打 0/1

judge_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是检索评估器。给定问题和一个候选证据片段，判断该片段是否对回答问题有直接帮助。\n"
            "只输出 JSON：{{\"relevant\": 0|1, \"reason\": \"...\"}}。",
        ),
        ("human", "问题：{q}\n\n证据片段：\n{c}"),
    ]
)


def is_relevant(q: str, chunk: str) -> int:
    raw = llm.invoke(judge_prompt.format_messages(q=q, c=chunk[:1200])).content
    j = parse_json_object(raw, {"relevant": 0, "reason": ""})
    return 1 if int(j.get("relevant", 0)) == 1 else 0


In [5]:
# 2) 召回评估：Recall@k（是否“在 top-k 里至少有 1 条相关证据”）


def recall_at_k(binary_labels: list[int]) -> float:
    # 假设只关心 top-k 中是否命中相关证据
    return 1.0 if any(binary_labels) else 0.0


all_recalls = []
for q in EVAL_QUERIES:
    docs = retrieve(q, k=TOPK)
    labels = [is_relevant(q, d.page_content) for d in docs]
    r = recall_at_k(labels)
    all_recalls.append(r)
    print("-", q)
    print("  labels:", labels, "Recall@k=", r)

print("\nAvg Recall@k:", sum(all_recalls) / len(all_recalls))


- 道通2024年年报里，主营业务/产品线的收入结构如何？
  labels: [1, 1, 0, 0, 1, 0, 0, 1] Recall@k= 1.0
- 年报里研发投入的主要方向是什么？
  labels: [0, 0, 0, 0, 0, 0, 1, 0] Recall@k= 1.0
- 年报里提到的海外/国际化业务有哪些信息？
  labels: [1, 0, 0, 1, 1, 0, 1, 0] Recall@k= 1.0
- 年报里有哪些风险因素或风险提示？
  labels: [1, 1, 0, 0, 1, 0, 0, 0] Recall@k= 1.0

Avg Recall@k: 1.0


In [6]:
# 3) 排序评估：MRR / nDCG


def mrr(labels: list[int]) -> float:
    for i, rel in enumerate(labels, 1):
        if rel:
            return 1.0 / i
    return 0.0


def ndcg(labels: list[int]) -> float:
    # binary relevance
    dcg = 0.0
    for i, rel in enumerate(labels, 1):
        if rel:
            dcg += 1.0 / math.log2(i + 1)
    # ideal DCG
    ideal = sorted(labels, reverse=True)
    idcg = 0.0
    for i, rel in enumerate(ideal, 1):
        if rel:
            idcg += 1.0 / math.log2(i + 1)
    return (dcg / idcg) if idcg > 0 else 0.0


rows = []
for q in EVAL_QUERIES:
    docs = retrieve(q, k=TOPK)
    labels = [is_relevant(q, d.page_content) for d in docs]
    rows.append((q, labels, mrr(labels), ndcg(labels)))

print("MRR avg:", sum(r[2] for r in rows) / len(rows))
print("nDCG avg:", sum(r[3] for r in rows) / len(rows))


MRR avg: 0.7857142857142857
nDCG avg: 0.7576857255030087


In [7]:
# 4) 对照：LLM Rerank（把 top-k 重新排序），再算 MRR / nDCG

rerank_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 reranker。对每个候选证据片段打 0-3 分（越高越相关）。\n"
            "只输出 JSON：{{\"scores\": [0|1|2|3, ...]}}，长度必须等于候选数。",
        ),
        ("human", "问题：{q}\n\n候选：\n{cands}"),
    ]
)


def llm_rerank(q: str, docs):
    cands = "\n\n".join([f"[{i}] {d.page_content[:450]}" for i, d in enumerate(docs, 1)])
    raw = llm.invoke(rerank_prompt.format_messages(q=q, cands=cands)).content
    parsed = parse_json_object(raw, {"scores": [0] * len(docs)})
    scores = parsed.get("scores", [0] * len(docs))
    if not isinstance(scores, list) or len(scores) != len(docs):
        scores = [0] * len(docs)

    order = sorted(range(len(docs)), key=lambda i: scores[i], reverse=True)
    return [docs[i] for i in order], scores


mrr0, ndcg0, mrr1, ndcg1 = [], [], [], []
for q in EVAL_QUERIES:
    docs = retrieve(q, k=TOPK)
    labels0 = [is_relevant(q, d.page_content) for d in docs]
    mrr0.append(mrr(labels0))
    ndcg0.append(ndcg(labels0))

    reranked, scores = llm_rerank(q, docs)
    labels1 = [is_relevant(q, d.page_content) for d in reranked]
    mrr1.append(mrr(labels1))
    ndcg1.append(ndcg(labels1))

    print("-", q)
    print("  scores:", scores)
    print("  before labels:", labels0, "MRR=", round(mrr0[-1], 3), "nDCG=", round(ndcg0[-1], 3))
    print("  after  labels:", labels1, "MRR=", round(mrr1[-1], 3), "nDCG=", round(ndcg1[-1], 3))

print("\nAvg MRR  before:", sum(mrr0) / len(mrr0), "after:", sum(mrr1) / len(mrr1))
print("Avg nDCG before:", sum(ndcg0) / len(ndcg0), "after:", sum(ndcg1) / len(ndcg1))


- 道通2024年年报里，主营业务/产品线的收入结构如何？
  scores: [2, 3, 0, 0, 3, 1, 1, 2]
  before labels: [1, 1, 0, 0, 1, 0, 0, 1] MRR= 1.0 nDCG= 0.911
  after  labels: [1, 1, 1, 1, 0, 0, 0, 0] MRR= 1.0 nDCG= 1.0
- 年报里研发投入的主要方向是什么？
  scores: [1, 1, 1, 0, 1, 0, 1, 1]
  before labels: [0, 0, 0, 0, 0, 0, 1, 0] MRR= 0.143 nDCG= 0.333
  after  labels: [0, 0, 0, 0, 1, 0, 0, 0] MRR= 0.2 nDCG= 0.387
- 年报里提到的海外/国际化业务有哪些信息？
  scores: [3, 0, 1, 3, 2, 1, 3, 1]
  before labels: [1, 0, 0, 1, 1, 0, 1, 0] MRR= 1.0 nDCG= 0.84
  after  labels: [1, 1, 1, 1, 0, 0, 0, 0] MRR= 1.0 nDCG= 1.0
- 年报里有哪些风险因素或风险提示？
  scores: [3, 2, 1, 0, 2, 0, 1, 0]
  before labels: [1, 1, 0, 0, 1, 0, 0, 0] MRR= 1.0 nDCG= 0.947
  after  labels: [1, 1, 1, 0, 0, 0, 0, 0] MRR= 1.0 nDCG= 1.0

Avg MRR  before: 0.7857142857142857 after: 0.8
Avg nDCG before: 0.7576857255030087 after: 0.8467132018086354


In [ ]:
# 5) RAGAS：端到端评估（检索 + 生成）
# 指标：answer_relevancy / faithfulness / context_precision / context_recall 等

from ragas import evaluate
from ragas.metrics import (
    answer_relevancy,
    faithfulness,
    context_precision,
    context_recall,
)

from datasets import Dataset

answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是基于证据回答问题的助手。只能使用给定上下文回答；若上下文不足，明确说不足。",
        ),
        ("human", "问题：{q}\n\n上下文：\n{ctx}"),
    ]
)

self_route_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 Self-RAG 的路由器。判断回答这个问题是否需要检索外部知识库。\n"
            "只输出 JSON：{{\"need_retrieve\": 0|1, \"reason\": \"...\"}}。",
        ),
        ("human", "问题：{q}"),
    ]
)

self_critique_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 Self-RAG 的 critique 模块。\n"
            "给定问题、草稿答案、以及检索证据（可能为空），判断草稿是否被证据支持。\n"
            "只输出 JSON：{{\"supported\": 0|1, \"reason\": \"...\", \"rewrite\": \"...\"}}。\n"
            "- supported=0 时给一个更利于检索的 rewrite（中文）。",
        ),
        ("human", "问题：{q}\n\n草稿：{draft}\n\n证据：\n{evidence}"),
    ]
)


def build_context(docs) -> tuple[str, list[str]]:
    ctxs = [d.page_content for d in docs]
    ctx = "\n\n".join(f"[{i}] {c}" for i, c in enumerate(ctxs, 1))
    return ctx, ctxs


def rag_answer(q: str, k: int = 6) -> tuple[str, list[str]]:
    docs = retrieve(q, k=k)
    ctx, ctxs = build_context(docs)
    ans = llm.invoke(answer_prompt.format_messages(q=q, ctx=ctx)).content.strip()
    return ans, ctxs


def self_rag_answer(q: str, k: int = 6) -> tuple[str, list[str]]:
    route_raw = llm.invoke(self_route_prompt.format_messages(q=q)).content
    route = parse_json_object(route_raw, {"need_retrieve": 1, "reason": ""})
    need_retrieve = 1 if int(route.get("need_retrieve", 1)) == 1 else 0

    docs = retrieve(q, k=k) if need_retrieve else []
    ctx, ctxs = build_context(docs)
    draft = llm.invoke(answer_prompt.format_messages(q=q, ctx=ctx)).content.strip()

    critique_raw = llm.invoke(
        self_critique_prompt.format_messages(q=q, draft=draft, evidence=ctx)
    ).content
    critique = parse_json_object(critique_raw, {"supported": 0, "reason": "", "rewrite": ""})
    supported = 1 if int(critique.get("supported", 0)) == 1 else 0
    rewrite = critique.get("rewrite", "")

    if supported:
        return draft, ctxs

    followup_query = rewrite or q
    docs2 = retrieve(followup_query, k=k)
    ctx2, ctxs2 = build_context(docs2)
    final = llm.invoke(answer_prompt.format_messages(q=q, ctx=ctx2)).content.strip()
    return final, ctxs2


def evaluate_strategy(name: str, answer_fn, k: int = 6):
    records = []
    for q in EVAL_QUERIES:
        ans, ctxs = answer_fn(q, k=k)
        records.append(
            {
                "question": q,
                "answer": ans,
                "contexts": ctxs,
                # 课堂版没有人工 ground_truth，这里先留空；如果你有标准答案集，填到这里
                "ground_truth": "",
            }
        )

    ds = Dataset.from_list(records)
    result = evaluate(
        ds,
        metrics=[answer_relevancy, faithfulness, context_precision, context_recall],
    )
    print(f"=== {name} ===")
    print(result)
    return result


baseline_result = evaluate_strategy("baseline_rag", rag_answer, k=6)
self_rag_result = evaluate_strategy("self_rag", self_rag_answer, k=6)


/opt/anaconda3/envs/voc/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/anaconda3/envs/voc/lib/python3.11/site-packages/instructor/providers/gemini/client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai  # type: ignore[import-not-found]
/var/folders/hb/4k5shxzs4c7by7lm5s0mmgrw0000gn/T/ipykernel_7870/3231578621.py:5: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.met

=== baseline_rag ===
{'answer_relevancy': nan, 'faithfulness': 0.8929, 'context_precision': 0.0000, 'context_recall': nan}


Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
The LLM did not return a valid classification.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
The LLM did not return a valid classification.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
The LLM did not return a valid classification.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
The LLM did not return a valid classification.
Exception raised in Job[0]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
Exception raised in Job[1]: InstructorRetryException(<failed_attempts>

<generation number="1">
<exception>
    The output is incomplete due to a max_tokens length limit.
</exception>
<completion>
    ChatCompletion(id='gen-1773379394-c924ITyjt29LMZwusLwk', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=Cha

=== self_rag ===
{'answer_relevancy': nan, 'faithfulness': 1.0000, 'context_precision': 0.0000, 'context_recall': nan}


## 结果解读

- `baseline_rag`：单次检索 + 单次生成。
- `self_rag`：先判断是否检索，再对草稿做 critique；若证据不足，则 rewrite 后补检索一次。
- 重点对比 `context_precision` 和 `faithfulness`，看 Self-RAG 是否减少了“拿到不够准的上下文就直接回答”的情况。